# Neo4j MoTrPAC Node Ingestion

Thiss notebook contains the beginnings of scripts used to convert the metadata and edge information found in the `../../kg/adjacency/` directory into Neo4j node and relation entities.

This notebook was originally run with Python version 3.14.7 in a Miniconda environment with ipykernel,pandas, and tqdm (Run `pip install -r requirements.txt` in an environment with Python 3.14.7 to set up accordingly).

Metadata has been downloaded from the [MoTrPAC Data Portal](https://motrpac-data.org/data-download/file-browser), but can also be accessed via the R packages found on the [MoTrPAC Github page](https://github.com/MoTrPAC).

In [1]:
import os
import pandas as pd
from tqdm import tqdm

## Nodes

### Tissues

In [2]:
tissues = pd.read_csv('../tissues.csv')[['tissue_id','species','tissue_name']]
print("CREATE")
print(',\n'.join([f"({tissue.iloc[0].replace(':','_').replace("-","_")}:Tissue {{tissue_id:'{tissue.iloc[0]}',species:'{tissue.iloc[1]}',name:'{tissue.iloc[2]}'}})" for _,tissue in tissues.iterrows()]), end=';')

CREATE
(human_muscle:Tissue {tissue_id:'human:muscle',species:'Human',name:'Skeletal muscle'}),
(human_adipose:Tissue {tissue_id:'human:adipose',species:'Human',name:'Adipose'}),
(human_plasma:Tissue {tissue_id:'human:plasma',species:'Human',name:'Plasma'}),
(human_whole_blood:Tissue {tissue_id:'human:whole-blood',species:'Human',name:'Whole blood'}),
(rat_ADRNL:Tissue {tissue_id:'rat:ADRNL',species:'Rat',name:'Adrenal gland'}),
(rat_BAT:Tissue {tissue_id:'rat:BAT',species:'Rat',name:'Brown adipose'}),
(rat_BLOOD:Tissue {tissue_id:'rat:BLOOD',species:'Rat',name:'Blood'}),
(rat_COLON:Tissue {tissue_id:'rat:COLON',species:'Rat',name:'Colon'}),
(rat_CORTEX:Tissue {tissue_id:'rat:CORTEX',species:'Rat',name:'Cerebral cortex'}),
(rat_HEART:Tissue {tissue_id:'rat:HEART',species:'Rat',name:'Heart'}),
(rat_HIPPOC:Tissue {tissue_id:'rat:HIPPOC',species:'Rat',name:'Hippocampus'}),
(rat_HYPOTH:Tissue {tissue_id:'rat:HYPOTH',species:'Rat',name:'Hypothalamus'}),
(rat_KIDNEY:Tissue {tissue_id:'rat:KI

### Genes

In [11]:
# This should point to the directory containing the TRANSCRIPTOMICS FEATURE METADATA files
transciptomics_dir = 'transcriptomics/metadata'

tdf = pd.concat([
    pd.read_csv(f'{transciptomics_dir}/{file}', sep='\t') for file in os.listdir(transciptomics_dir)
]).drop_duplicates().reset_index(drop=True).set_index('feature_id')
gene_ids = tdf.set_index('gene_symbol')['entrez_gene'].dropna().astype(int).to_dict()
tdf

,assay,entrez_gene,gene_symbol,ensembl_gene
feature_id,,,,
ENSG00000000419.14,transcript-rna-seq,8813.0,DPM1,ENSG00000000419
ENSG00000000457.14,transcript-rna-seq,57147.0,SCYL3,ENSG00000000457
ENSG00000000460.17,transcript-rna-seq,55732.0,C1orf112,ENSG00000000460
ENSG00000000938.13,transcript-rna-seq,2268.0,FGR,ENSG00000000938
ENSG00000000971.16,transcript-rna-seq,3075.0,CFH,ENSG00000000971
...,...,...,...,...
ENSG00000289345.1,transcript-rna-seq,NaN,NaN,ENSG00000289345
ENSG00000289428.1,transcript-rna-seq,NaN,NaN,ENSG00000289428
ENSG00000289466.1,transcript-rna-seq,NaN,NaN,ENSG00000289466


In [12]:
# Drop rows with no gene symbol (can't key a node without one), then dedupe on symbol
genes = tdf.dropna(subset=['gene_symbol']).drop_duplicates(subset='gene_symbol', keep='first')

lines = []
for _, gene in genes.iterrows():
    symbol = gene.gene_symbol
    props = [f"symbol:'{symbol}'"]
    if not pd.isna(gene.entrez_gene):
        props.append(f"ncbi:{int(gene.entrez_gene)}")
    if not pd.isna(gene.ensembl_gene):
        props.append(f"ensembl:'{gene.ensembl_gene}'")
    lines.append(f"(`{symbol}`:Gene {{{', '.join(props)}}})")

cypher = "CREATE\n" + ",\n".join(lines) + ";"
print(cypher)

CREATE
(`DPM1`:Gene {symbol:'DPM1', ncbi:8813, ensembl:'ENSG00000000419'}),
(`SCYL3`:Gene {symbol:'SCYL3', ncbi:57147, ensembl:'ENSG00000000457'}),
(`C1orf112`:Gene {symbol:'C1orf112', ncbi:55732, ensembl:'ENSG00000000460'}),
(`FGR`:Gene {symbol:'FGR', ncbi:2268, ensembl:'ENSG00000000938'}),
(`CFH`:Gene {symbol:'CFH', ncbi:3075, ensembl:'ENSG00000000971'}),
(`FUCA2`:Gene {symbol:'FUCA2', ncbi:2519, ensembl:'ENSG00000001036'}),
(`GCLC`:Gene {symbol:'GCLC', ncbi:2729, ensembl:'ENSG00000001084'}),
(`NFYA`:Gene {symbol:'NFYA', ncbi:4800, ensembl:'ENSG00000001167'}),
(`STPG1`:Gene {symbol:'STPG1', ncbi:90529, ensembl:'ENSG00000001460'}),
(`NIPAL3`:Gene {symbol:'NIPAL3', ncbi:57185, ensembl:'ENSG00000001461'}),
(`LAS1L`:Gene {symbol:'LAS1L', ncbi:81887, ensembl:'ENSG00000001497'}),
(`ENPP4`:Gene {symbol:'ENPP4', ncbi:22875, ensembl:'ENSG00000001561'}),
(`ANKIB1`:Gene {symbol:'ANKIB1', ncbi:54467, ensembl:'ENSG00000001629'}),
(`CYP51A1`:Gene {symbol:'CYP51A1', ncbi:1595, ensembl:'ENSG00000001

#### Subset

In [13]:
genes = pd.read_csv('../Genes.csv')
human_genes = genes[['human_symbol','human_gene_id']]
rat_genes = genes[['rat_symbol','rat_gene_id']]
print("CREATE")
print(',\n'.join([f"({gene.iloc[0]}:Gene {{symbol:'{gene.iloc[0]}','ncbi':{gene_ids[gene.iloc[0]]},'ensembl':{gene.iloc[1]}}})" for _,gene in human_genes.iterrows()]), end=',\n')
print(',\n'.join([f"({gene.iloc[0]}:Gene {{symbol:'{gene.iloc[0]}','ensembl':{gene.iloc[1]}}})" for _,gene in rat_genes.iterrows()]), end=';')

CREATE
(SLC7A5:Gene {symbol:'SLC7A5','ncbi':8140,'ensembl':ENSG00000103257}),
(SLC38A4:Gene {symbol:'SLC38A4','ncbi':55089,'ensembl':ENSG00000139209}),
(ALDH7A1:Gene {symbol:'ALDH7A1','ncbi':501,'ensembl':ENSG00000164904}),
(OARD1:Gene {symbol:'OARD1','ncbi':221443,'ensembl':ENSG00000124596}),
(PNP:Gene {symbol:'PNP','ncbi':4860,'ensembl':ENSG00000198805}),
(CHKB:Gene {symbol:'CHKB','ncbi':1120,'ensembl':ENSG00000100288}),
(BCAT2:Gene {symbol:'BCAT2','ncbi':587,'ensembl':ENSG00000105552}),
(AADAT:Gene {symbol:'AADAT','ncbi':51166,'ensembl':ENSG00000109576}),
(GPCPD1:Gene {symbol:'GPCPD1','ncbi':56261,'ensembl':ENSG00000125772}),
(ETNK1:Gene {symbol:'ETNK1','ncbi':55500,'ensembl':ENSG00000139163}),
(Slc7a5:Gene {symbol:'Slc7a5','ensembl':ENSRNOG00000018824}),
(Slc38a4:Gene {symbol:'Slc38a4','ensembl':ENSRNOG00000006653}),
(Aldh7a1:Gene {symbol:'Aldh7a1','ensembl':ENSRNOG00000014645}),
(Oard1:Gene {symbol:'Oard1','ensembl':ENSRNOG00000012679}),
(Pnp:Gene {symbol:'Pnp','ensembl':ENSRNOG00

### Metabolites

In [37]:
mets = pd.read_csv('../../kg/adjacency/human_metab_contrast_adjacency_529x21_rows.csv',index_col=0)
mets

,metabolite_id,refmet_name,platform,shared_with_other_species,n_cells,best_adj_p
row,,,,,,
Xanthosine,REFMET:Xanthosine,Xanthosine,metab-u-hilicpos,True,9,7.194406e-99
Hypoxanthine,REFMET:Hypoxanthine,Hypoxanthine,meta-reg,True,8,2.798199e-86
Malic acid,REFMET:Malic acid,Malic acid,meta-reg,True,6,2.226413e-69
Lactic acid,REFMET:Lactic acid,Lactic acid,meta-reg,True,7,2.829914e-64
Succinic acid,REFMET:Succinic acid,Succinic acid,metab-t-tca,True,6,3.342169e-60
...,...,...,...,...,...,...
PC 35:3,REFMET:PC 35:3,PC 35:3,metab-u-rppos,True,1,4.754576e-02
Methylimidazoleacetic acid,REFMET:Methylimidazoleacetic acid,Methylimidazoleacetic acid,metab-u-hilicpos,True,1,4.768983e-02
CAR 20:0,REFMET:CAR 20:0,CAR 20:0,metab-u-rppos,True,1,4.803460e-02


In [ ]:
# Process an example subset of metabolites, can be extended for eventual larger KG
metabolites = mets.iloc[:10]
metabolites

,metabolite_id,refmet_name,platform,shared_with_other_species,n_cells,best_adj_p
row,,,,,,
Xanthosine,REFMET:Xanthosine,Xanthosine,metab-u-hilicpos,True,9,7.194406e-99
Hypoxanthine,REFMET:Hypoxanthine,Hypoxanthine,meta-reg,True,8,2.798199e-86
Malic acid,REFMET:Malic acid,Malic acid,meta-reg,True,6,2.226413e-69
Lactic acid,REFMET:Lactic acid,Lactic acid,meta-reg,True,7,2.829914e-64
Succinic acid,REFMET:Succinic acid,Succinic acid,metab-t-tca,True,6,3.342169e-60
Xanthine,REFMET:Xanthine,Xanthine,metab-u-hilicpos,True,7,1.603280e-59
Alanine,REFMET:Alanine,Alanine,meta-reg,True,6,5.500408e-58
Inosine,REFMET:Inosine,Inosine,meta-reg,True,6,3.243030e-52
Pantothenic acid,REFMET:Pantothenic acid,Pantothenic acid,meta-reg,True,7,3.853924e-46


In [40]:
"""
Build Cypher CREATE statements for :Metabolite nodes from a table shaped like:

    row              metabolite_id             refmet_name        platform            shared_with_other_species  n_cells  best_adj_p
    Xanthosine       REFMET:Xanthosine         Xanthosine         metab-u-hilicpos    True                       9        7.194406e-99
    Hypoxanthine     REFMET:Hypoxanthine       Hypoxanthine       meta-reg            True                       8        2.798199e-86
    ...

Only three properties are kept on the node, per the request:
  - metabolite_name  <- the 'row' index (e.g. 'Xanthosine')
  - metabolite_id    <- the 'metabolite_id' column (e.g. 'REFMET:Xanthosine')
  - refmet_name       <- the 'refmet_name' column (e.g. 'Xanthosine')

platform / shared_with_other_species / n_cells / best_adj_p are ignored,
since they weren't asked for -- add them below if you want them kept too.
"""

import re
import pandas as pd

_VAR_SANITIZE_RE = re.compile(r'[^0-9a-zA-Z_]')


def cypher_var(name: str, used: set) -> str:
    """Turn a metabolite name into a unique, valid Cypher variable name."""
    var = _VAR_SANITIZE_RE.sub('_', str(name)).strip('_')
    if not var or var[0].isdigit():
        var = f'm_{var}'
    base = var
    i = 2
    while var in used:
        var = f'{base}_{i}'
        i += 1
    used.add(var)
    return var


def escape(value: str) -> str:
    """Escape single quotes for a Cypher string literal."""
    return str(value).replace("\\", "\\\\").replace("'", "\\'")


def build_create_statements(df: pd.DataFrame) -> list[str]:
    """
    df: the metabolite table with the metabolite name as the index (or in a
    'row' column) and 'metabolite_id' / 'refmet_name' columns, as shown above.
    """
    df = df.copy()
    if 'row' in df.columns:
        df = df.set_index('row')
    df = df.rename_axis('row')

    used_vars = set()
    statements = []

    for metabolite_name, r in df.iterrows():
        var = cypher_var(metabolite_name, used_vars)
        metabolite_id = r['metabolite_id']
        refmet_name = r['refmet_name']

        statement = (
            f"CREATE ({var}:Metabolite {{"
            f"metabolite_id:'{escape(metabolite_id)}',"
            f"refmet_name:'{escape(refmet_name)}',"
            f"metabolite_name:'{escape(metabolite_name)}'"
            f"}})"
        )
        statements.append(statement)

    return statements



df = metabolites

statements = build_create_statements(df)

    # One CREATE per line, comma-joined so it can also be run as a
    # single multi-node CREATE statement if you prefer -- swap
    # ',\n'.join(...) + ';' for '\n'.join(s + ';' for s in statements)
    # if you'd rather have one CREATE per line instead.
print(';\n'.join(statements) + ';\n')

CREATE (Xanthosine:Metabolite {metabolite_id:'REFMET:Xanthosine',refmet_name:'Xanthosine',metabolite_name:'Xanthosine'});
CREATE (Hypoxanthine:Metabolite {metabolite_id:'REFMET:Hypoxanthine',refmet_name:'Hypoxanthine',metabolite_name:'Hypoxanthine'});
CREATE (Malic_acid:Metabolite {metabolite_id:'REFMET:Malic acid',refmet_name:'Malic acid',metabolite_name:'Malic acid'});
CREATE (Lactic_acid:Metabolite {metabolite_id:'REFMET:Lactic acid',refmet_name:'Lactic acid',metabolite_name:'Lactic acid'});
CREATE (Succinic_acid:Metabolite {metabolite_id:'REFMET:Succinic acid',refmet_name:'Succinic acid',metabolite_name:'Succinic acid'});
CREATE (Xanthine:Metabolite {metabolite_id:'REFMET:Xanthine',refmet_name:'Xanthine',metabolite_name:'Xanthine'});
CREATE (Alanine:Metabolite {metabolite_id:'REFMET:Alanine',refmet_name:'Alanine',metabolite_name:'Alanine'});
CREATE (Inosine:Metabolite {metabolite_id:'REFMET:Inosine',refmet_name:'Inosine',metabolite_name:'Inosine'});
CREATE (Pantothenic_acid:Metabol

## Relations

### Gene-Tissue

In [ ]:
# Computing relations for subset of genes, could do for all relations for full KG
relations = pd.read_csv('../../kg/adjacency/gene_tissue_adjacency_15894x23.csv', index_col=0)
relations_subset = relations[relations.index.isin(human_genes['human_symbol'])]
relations_subset

,human:muscle,human:adipose,human:blood,rat:ADRNL,rat:BAT,rat:BLOOD,rat:COLON,rat:CORTEX,rat:HEART,rat:HIPPOC,...,rat:LUNG,rat:OVARY,rat:PLASMA,rat:SKM-GN,rat:SKM-VL,rat:SMLINT,rat:SPLEEN,rat:TESTES,rat:VENACV,rat:WAT-SC
row,,,,,,,,,,,,,,,,,,,,,
AADAT,-11.980,0.0,0.000,1.815,0.000,0.000,0.000,0.0,0.000,0.000,...,0.00,0.000,0.0,0.000,0.000,0.0,0.000,0.0,0.0,2.426
ALDH7A1,-2.060,0.0,0.000,1.751,-11.750,0.000,4.260,0.0,0.000,0.000,...,0.00,0.000,0.0,0.000,0.000,0.0,0.000,0.0,0.0,0.000
BCAT2,-3.365,0.0,-1.371,0.000,0.000,0.000,3.318,0.0,0.000,0.000,...,0.00,0.000,0.0,-3.550,-10.840,0.0,0.000,0.0,0.0,0.000
GPCPD1,9.632,0.0,5.894,0.000,0.000,4.043,0.000,0.0,2.847,0.000,...,1.64,0.000,0.0,4.890,-2.110,0.0,2.509,0.0,0.0,0.000
SLC7A5,9.294,0.0,0.000,0.000,2.396,2.804,2.826,0.0,0.000,0.000,...,0.00,0.000,0.0,0.000,0.000,0.0,2.198,0.0,0.0,0.000
PNP,4.827,0.0,-1.469,1.586,0.000,0.000,0.000,0.0,0.000,0.000,...,0.00,-6.817,0.0,0.000,0.000,0.0,3.075,0.0,0.0,1.506
CHKB,-5.521,0.0,0.000,-5.258,1.849,0.000,0.000,0.0,-1.679,0.000,...,0.00,0.000,0.0,0.000,0.000,0.0,-1.681,0.0,0.0,0.000
OARD1,-3.976,0.0,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,...,0.00,0.000,0.0,-3.118,0.000,0.0,0.000,0.0,0.0,0.000
SLC38A4,-3.824,0.0,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,...,0.00,0.000,0.0,0.000,-1.392,0.0,0.000,0.0,0.0,0.000


In [26]:
ASSAY = 'transcript-rna-seq'
MEASUREMENT = 'RNA'
TISSUE_ID_OVERRIDES = {
    'human:blood': 'human:whole-blood',
}
 
def rat_symbol(human_symbol: str) -> str:
    """Convert a human gene symbol to its rat-node symbol form."""
    return human_symbol.capitalize()
 
def build_queries(df: pd.DataFrame) -> list[str]:
    """
    df must have three columns in this order: [human_gene_symbol, tissue_key, score]
    exactly as produced by:
        relations_subset[relations_subset!=0].stack().dropna().reset_index()
    """
    df = df.copy()
    df.columns = ['gene', 'tissue_key', 'score']
 
    queries = []
    unmatched_tissue_keys = set()
 
    for _, r in df.iterrows():
        human_gene = str(r['gene'])
        tissue_key = str(r['tissue_key'])
        score = r['score']
 
        tissue_id = TISSUE_ID_OVERRIDES.get(tissue_key, tissue_key)
        is_rat_tissue = tissue_id.startswith('rat:')
 
        gene_symbol = rat_symbol(human_gene) if is_rat_tissue else human_gene
 
        if tissue_key not in TISSUE_ID_OVERRIDES and tissue_key.startswith('human:') and tissue_key not in {
            'human:muscle', 'human:adipose', 'human:plasma', 'human:whole-blood'
        }:
            unmatched_tissue_keys.add(tissue_key)
 
        query = (
            f"MATCH (a:Gene {{symbol:'{gene_symbol}'}}),(b:Tissue {{tissue_id:'{tissue_id}'}}) "
            f"MERGE (a)-[:EXPRESSED {{assay:'{ASSAY}',measurement:'{MEASUREMENT}',score:{score}}}]->(b);"
        )
        queries.append(query)
 
    if unmatched_tissue_keys:
        print("WARNING: these tissue keys don't match any known Tissue node "
              "and have no override defined -- add them to TISSUE_ID_OVERRIDES:")
        for k in sorted(unmatched_tissue_keys):
            print(f"  - {k}")
 
    return queries
 

df = relations_subset[relations_subset != 0].stack().dropna().reset_index()

queries = build_queries(df)

out_path = 'gene_tissue.cypher'
with open(out_path, 'w') as f:
    f.write('\n'.join(queries) + '\n')

print(f"Wrote {len(queries)} EXPRESSED relationship queries to {out_path}")

Wrote 51 EXPRESSED relationship queries to gene_tissue.cypher


### Gene-Tissue Contrast

In [28]:
# Computing relations for subset of genes, could do for all relations for full KG
relations = pd.read_csv('../../kg/adjacency/human_gene_contrast_adjacency_12815x21.csv', index_col=0)
contrast_relations_subset = relations[relations.index.isin(human_genes['human_symbol'])]
contrast_relations_subset

,adipose|EE-CON|post_15_30_45_min,adipose|EE-CON|post_3.5_4_hr,adipose|EE-CON|post_24_hr,adipose|RE-CON|post_15_30_45_min,adipose|RE-CON|post_3.5_4_hr,blood|EE-CON|during_20_min,blood|EE-CON|during_40_min,blood|EE-CON|post_10_min,blood|EE-CON|post_15_30_45_min,blood|EE-CON|post_3.5_4_hr,...,blood|RE-CON|post_10_min,blood|RE-CON|post_15_30_45_min,blood|RE-CON|post_3.5_4_hr,blood|RE-CON|post_24_hr,muscle|EE-CON|post_15_30_45_min,muscle|EE-CON|post_3.5_4_hr,muscle|EE-CON|post_24_hr,muscle|RE-CON|post_15_30_45_min,muscle|RE-CON|post_3.5_4_hr,muscle|RE-CON|post_24_hr
row,,,,,,,,,,,,,,,,,,,,,
AADAT,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.00,0.0,0.000,...,0.0,0.0,0.000,0.0,-1.615,-6.852,0.000,0.000,-11.980,0.000
GPCPD1,0.0,0.0,0.0,0.0,0.0,-1.541,-2.773,-1.88,0.0,2.222,...,0.0,0.0,5.894,0.0,0.000,7.290,0.000,1.359,9.632,0.000
SLC7A5,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.00,0.0,0.000,...,0.0,0.0,0.000,0.0,0.000,5.634,0.000,0.000,9.294,-2.076
CHKB,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.00,0.0,0.000,...,0.0,0.0,0.000,0.0,-5.521,0.000,2.819,-1.753,-5.237,0.000
PNP,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.00,0.0,0.000,...,0.0,0.0,-1.469,0.0,4.827,3.094,0.000,2.997,2.686,0.000
OARD1,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.00,0.0,0.000,...,0.0,0.0,0.000,0.0,-1.547,0.000,1.526,-2.313,-3.976,0.000
SLC38A4,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.00,0.0,0.000,...,0.0,0.0,0.000,0.0,0.000,-3.211,0.000,0.000,-3.590,-3.824
BCAT2,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.00,0.0,-1.371,...,0.0,0.0,0.000,0.0,-2.122,0.000,0.000,0.000,0.000,-3.365
ETNK1,0.0,0.0,0.0,0.0,0.0,-2.040,0.000,0.00,0.0,0.000,...,0.0,0.0,0.000,0.0,2.554,0.000,0.000,2.268,0.000,0.000


In [29]:
"""
Build Cypher MERGE statements for (:Gene)-[:EXERCISE_EXPRESSED]->(:Tissue)
relationships from a contrast matrix shaped like:

    row                adipose|EE-CON|post_15_30_45_min   blood|EE-CON|during_20_min   ...
    AADAT              0.0                                 0.000                       ...
    GPCPD1             0.0                                 -1.541                      ...
    ...

Column names are pipe-delimited as: "<tissue>|<condition>|<time>", e.g.
'muscle|RE-CON|post_3.5_4_hr' -> tissue='muscle', condition='RE-CON',
time='post_3.5_4_hr'.

All gene symbols in this matrix are the HUMAN form (e.g. 'AADAT'), so no
species symbol-swapping is needed here (unlike the rat/human EXPRESSED
script) -- every row matches a human Gene node directly.
"""

import pandas as pd

ASSAY = 'transcript-rna-seq'
MEASUREMENT = 'RNA'

TISSUE_SHORT_MAP = {
    'adipose': 'human:adipose',
    'blood': 'human:whole-blood',
    'muscle': 'human:muscle',
}


def parse_column(col: str):
    """Split a 'tissue|condition|time' column name into its three parts."""
    parts = col.split('|')
    if len(parts) != 3:
        raise ValueError(f"Unexpected column format (expected tissue|condition|time): {col!r}")
    tissue, condition, time = parts
    return tissue, condition, time


def build_queries(df: pd.DataFrame) -> list[str]:
    """
    df: the contrast matrix with genes as the index (or a 'row' column) and
    'tissue|condition|time' column names, as shown above.
    """
    df = df.copy()
    if df.index.name == 'row' or df.index.name is None:
        # Ensure the gene symbol is a normal column named 'row'
        df = df.reset_index().rename(columns={'index': 'row'})
        if 'row' not in df.columns:
            df = df.rename(columns={df.columns[0]: 'row'})

    stacked = (
        df.set_index('row')
        .stack()
        .replace(0, pd.NA)
        .dropna()
        .reset_index()
    )
    stacked.columns = ['gene', 'column', 'score']

    queries = []
    unmatched_tissues = set()

    for _, r in stacked.iterrows():
        gene_symbol = str(r['gene'])
        tissue_short, condition, time = parse_column(str(r['column']))
        score = r['score']

        tissue_id = TISSUE_SHORT_MAP.get(tissue_short)
        if tissue_id is None:
            unmatched_tissues.add(tissue_short)
            continue

        query = (
            f"MATCH (a:Gene {{symbol:'{gene_symbol}'}}),(b:Tissue {{tissue_id:'{tissue_id}'}}) "
            f"MERGE (a)-[:EXERCISE_EXPRESSED {{assay:'{ASSAY}',measurement:'{MEASUREMENT}',"
            f"score:{score},condition:'{condition}',time:'{time}'}}]->(b);"
        )
        queries.append(query)

    if unmatched_tissues:
        print("WARNING: these tissue short-names have no entry in TISSUE_SHORT_MAP "
              "and were skipped:")
        for t in sorted(unmatched_tissues):
            print(f"  - {t}")

    return queries



df = contrast_relations_subset

queries = build_queries(df)

out_path = 'gene_tissue_contrast.cypher'
with open(out_path, 'w') as f:
    f.write('\n'.join(queries) + '\n')

print(f"Wrote {len(queries)} EXERCISE_EXPRESSED relationship queries to {out_path}")

Wrote 38 EXERCISE_EXPRESSED relationship queries to gene_tissue_contrast.cypher


In [31]:
# Computing relations for subset of genes, could do for all relations for full KG
relations = pd.read_csv('../../kg/adjacency/rat_gene_contrast_adjacency_11283x150.csv', index_col=0)
contrast_relations_subset = relations[relations.index.isin(rat_genes['rat_symbol'])]
contrast_relations_subset

,ADRNL|F|1w,ADRNL|F|2w,ADRNL|F|4w,ADRNL|F|8w,ADRNL|M|1w,ADRNL|M|2w,ADRNL|M|4w,ADRNL|M|8w,BAT|F|1w,BAT|F|2w,...,VENACV|M|4w,VENACV|M|8w,WAT-SC|F|1w,WAT-SC|F|2w,WAT-SC|F|4w,WAT-SC|F|8w,WAT-SC|M|1w,WAT-SC|M|2w,WAT-SC|M|4w,WAT-SC|M|8w
row,,,,,,,,,,,,,,,,,,,,,
Aldh7a1,-1.751,-1.751,-1.751,-1.751,1.751,0.0,0.0,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000
Bcat2,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000
Pnp,0.000,0.000,0.000,1.586,0.000,0.0,0.0,1.586,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.506,1.506,1.506,1.506
Chkb,-5.258,-5.258,-5.258,-5.258,5.258,0.0,0.0,0.000,1.849,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000
Gpcpd1,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000
Etnk1,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000
Oard1,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000
Slc7a5,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,2.396,2.396,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000
Aadat,1.815,1.815,1.815,0.000,0.000,0.0,0.0,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,2.426,2.426,2.426,2.426


In [33]:
"""
Build Cypher MERGE statements for (:Gene)-[:EXERCISE_EXPRESSED]->(:Tissue)
relationships from the rat contrast matrix, shaped like:

    row       ADRNL|F|1w   ADRNL|F|2w   ...   WAT-SC|M|8w
    Aldh7a1   -1.751       -1.751       ...   0.000
    Bcat2      0.000        0.000       ...   0.000
    ...

Column names are pipe-delimited as: "<tissue>|<sex>|<time>", e.g.
'WAT-SC|M|8w' -> tissue='WAT-SC', sex='M', time='8w'.

Genes here are already in the rat symbol form (e.g. 'Aldh7a1'), and the
tissue codes (ADRNL, BAT, WAT-SC, ...) match the rat Tissue node tissue_id
suffixes directly -- just prefix with 'rat:'. So, unlike the human/rat
EXPRESSED script, no symbol-casing conversion is needed here.
"""

import pandas as pd

ASSAY = 'transcript-rna-seq'
MEASUREMENT = 'RNA'


def parse_column(col: str):
    """Split a 'tissue|sex|time' column name into its three parts."""
    parts = col.split('|')
    if len(parts) != 3:
        raise ValueError(f"Unexpected column format (expected tissue|sex|time): {col!r}")
    tissue, sex, time = parts
    return tissue, sex, time


def build_queries(df: pd.DataFrame) -> list[str]:
    """
    df: the rat contrast matrix with gene symbols as the index (or a 'row'
    column) and 'tissue|sex|time' column names, as shown above.
    """
    df = df.copy()
    # Normalize to "gene symbol as the index" regardless of whether it
    # currently lives in the index or in a 'row' column, then stack directly
    # off the index (no round-trip through a same-named column, which is
    # what caused the earlier ('Aldh7a1', -1.751)-style tuple symbols).
    if 'row' in df.columns:
        df = df.set_index('row')
    df = df.rename_axis('row')

    stacked = (
        df.stack()
        .replace(0, pd.NA)
        .dropna()
        .reset_index()
    )
    stacked.columns = ['gene', 'column', 'score']

    queries = []
    unmatched_tissues = set()

    for _, r in stacked.iterrows():
        gene_symbol = str(r['gene'])
        tissue_short, sex, time = parse_column(str(r['column']))
        score = r['score']

        tissue_id = f'rat:{tissue_short}'

        query = (
            f"MATCH (a:Gene {{symbol:'{gene_symbol}'}}),(b:Tissue {{tissue_id:'{tissue_id}'}}) "
            f"MERGE (a)-[:EXERCISE_EXPRESSED {{assay:'{ASSAY}',measurement:'{MEASUREMENT}',"
            f"score:{score},sex:'{sex}',time:'{time}'}}]->(b);"
        )
        queries.append(query)

    if unmatched_tissues:
        print("WARNING: these tissue codes had no match:")
        for t in sorted(unmatched_tissues):
            print(f"  - {t}")

    return queries



df = contrast_relations_subset

queries = build_queries(df)

out_path = 'rat_gene_tissue_contrast.cypher'
with open(out_path, 'w') as f:
    f.write('\n'.join(queries) + '\n')

print(f"Wrote {len(queries)} EXERCISE_EXPRESSED relationship queries to {out_path}")

Wrote 167 EXERCISE_EXPRESSED relationship queries to rat_gene_tissue_contrast.cypher


### Tissue-Metabolite

In [ ]:
# Computing relations for subset of genes, could do for all relations for full KG
relations = pd.read_csv('../../kg/adjacency/human_metab_contrast_adjacency_', index_col=0)
relations_subset = relations[relations.index.isin(rat_genes['rat_symbol'])]
relations_subset

## Scratch

### Epigenomics

In [35]:
epigenomics_dir = 'epigenomics/metadata'

edf = pd.concat([
    pd.read_csv(f'{epigenomics_dir}/{file}', sep='\t') for file in os.listdir(epigenomics_dir)
])[["assay","feature_id","entrez_gene","gene_symbol","ensembl_gene", "custom_annotation"]].drop_duplicates().reset_index(drop=True).set_index('feature_id')
edf

,assay,entrez_gene,gene_symbol,ensembl_gene,custom_annotation
feature_id,,,,,
chr10:100010277-100010277,epigen-methylcap-seq,23268.0,DNMBP,ENSG00000107554,Promoter (<=1kb)
chr10:100010333-100010333,epigen-methylcap-seq,23268.0,DNMBP,ENSG00000107554,Promoter (<=1kb)
chr10:100010346-100010346,epigen-methylcap-seq,23268.0,DNMBP,ENSG00000107554,Promoter (<=1kb)
chr10:100010373-100010373,epigen-methylcap-seq,23268.0,DNMBP,ENSG00000107554,Promoter (<=1kb)
chr10:100010407-100010407,epigen-methylcap-seq,23268.0,DNMBP,ENSG00000107554,Promoter (<=1kb)
...,...,...,...,...,...
chr9:99678622-99678878,epigen-atac-seq,101928438.0,NaN,ENSG00000237461,Intron
chr9:99703160-99703963,epigen-atac-seq,101928438.0,NaN,ENSG00000237461,Intron
chr9:99726617-99727341,epigen-atac-seq,101928438.0,NaN,ENSG00000237461,Intron


### Metabolomics

In [34]:
metabolomics_untargeted_dir = 'metabolomics-untargeted/metadata'
metabolomics_targeted_dir = 'metabolomics-targeted/metadata'

mdf = pd.concat([
    pd.read_csv(f'{metabolomics_untargeted_dir}/{file}', sep='\t') for file in os.listdir(metabolomics_untargeted_dir)
]+[
    pd.read_csv(f'{metabolomics_targeted_dir}/{file}', sep='\t') for file in os.listdir(metabolomics_targeted_dir)
])[["assay","feature_id","refmet_name","refmet_id","kegg_id"]].drop_duplicates().reset_index(drop=True).set_index('feature_id')
mdf

,assay,refmet_name,refmet_id,kegg_id
feature_id,,,,
1-Methylxanthine,metab,1-Methylxanthine,RM0136343,C16358
"1,3-Dimethyluric acid",metab,"1,3-Dimethyluric acid",RM0136120,NaN
1003.6721_11.476,metab,NaN,NaN,NaN
1004.6749_13.717,metab,NaN,NaN,NaN
1005.6876_9.866,metab,NaN,NaN,NaN
...,...,...,...,...
adenosine,metab,Adenosine,RM0135870,C00212
12-HETE,metab,12-HETE,RM0152612,NaN
12(13)-EpMOE,metab,12(13)-EpOME,RM0153719,C14826


### Proteomics

In [33]:
proteomics_dir = 'proteomics/metadata'

pdf = pd.concat([
    pd.read_csv(f'{proteomics_dir}/{file}', sep='\t') for file in os.listdir(proteomics_dir)
])[["assay","feature_id","entrez_gene","gene_symbol","ensembl_gene","uniprot"]].drop_duplicates().reset_index(drop=True).set_index('feature_id')
pdf

,assay,entrez_gene,gene_symbol,ensembl_gene,uniprot
feature_id,,,,,
A0A0B4J2F2_S575s,prot-ph,102724428.0,SIK1B,ENSG00000275993,A0A0B4J2F2
A0A0B4J2F2_T473t,prot-ph,102724428.0,SIK1B,ENSG00000275993,A0A0B4J2F2
A0A0U1RRK4_S41s,prot-ph,107984640.0,LBHD2,ENSG00000283071,A0A0U1RRK4
A0A1B0GTU1_S495s,prot-ph,643136.0,ZC3H11B,ENSG00000215817,A0A1B0GTU1
A0AVT1_S36s,prot-ph,55236.0,UBA6,ENSG00000033178,A0AVT1
...,...,...,...,...,...
OID21527,prot-ol,939.0,CD27,ENSG00000139193,P26842
OID21528,prot-ol,634.0,CEACAM1,ENSG00000079385,P13688
OID21529,prot-ol,2348.0,FOLR1,ENSG00000110195,P15328


### Transcriptomics

In [67]:
transciptomics_dir = 'transcriptomics/metadata'

tdf = pd.concat([
    pd.read_csv(f'{transciptomics_dir}/{file}', sep='\t') for file in os.listdir(transciptomics_dir)
]).drop_duplicates().reset_index(drop=True).set_index('feature_id')
tdf

,assay,entrez_gene,gene_symbol,ensembl_gene
feature_id,,,,
ENSG00000000419.14,transcript-rna-seq,8813.0,DPM1,ENSG00000000419
ENSG00000000457.14,transcript-rna-seq,57147.0,SCYL3,ENSG00000000457
ENSG00000000460.17,transcript-rna-seq,55732.0,C1orf112,ENSG00000000460
ENSG00000000938.13,transcript-rna-seq,2268.0,FGR,ENSG00000000938
ENSG00000000971.16,transcript-rna-seq,3075.0,CFH,ENSG00000000971
...,...,...,...,...
ENSG00000289345.1,transcript-rna-seq,NaN,NaN,ENSG00000289345
ENSG00000289428.1,transcript-rna-seq,NaN,NaN,ENSG00000289428
ENSG00000289466.1,transcript-rna-seq,NaN,NaN,ENSG00000289466


In [53]:
tdf.apply(lambda x: 
    f"{x.name}:Molecule{{type:'transcript',ncbi:{x.iloc[1]},symbol:{x.iloc[2]},ensembl:{x.iloc[3]}}}", axis=1
)

feature_id
ENSG00000000419.14    ENSG00000000419.14:Molecule{type:'transcript',...
ENSG00000000457.14    ENSG00000000457.14:Molecule{type:'transcript',...
ENSG00000000460.17    ENSG00000000460.17:Molecule{type:'transcript',...
ENSG00000000938.13    ENSG00000000938.13:Molecule{type:'transcript',...
ENSG00000000971.16    ENSG00000000971.16:Molecule{type:'transcript',...
                                            ...                        
ENSG00000289345.1     ENSG00000289345.1:Molecule{type:'transcript',n...
ENSG00000289428.1     ENSG00000289428.1:Molecule{type:'transcript',n...
ENSG00000289466.1     ENSG00000289466.1:Molecule{type:'transcript',n...
ENSG00000289609.1     ENSG00000289609.1:Molecule{type:'transcript',n...
ENSG00000289642.1     ENSG00000289642.1:Molecule{type:'transcript',n...
Length: 24007, dtype: str

In [57]:
template = f"""CREATE
{',\n'.join(tdf.apply(lambda x: 
    f"('{x.name}':Molecule{{type:'transcript',ncbi:{x.iloc[1]},symbol:{x.iloc[2]},ensembl:{x.iloc[3]}}})", axis=1
))};
"""
with open('transcript.cypher', 'w') as f:
    print(template, file=f)